# N029 · 可变窗口与最短覆盖

**目标：** 写出窗口有效条件并按方向收缩。

**先修：** N020, N028。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 最长无重复；覆盖频次；缺失计数；收缩条件；可行性单调前提。

**配套讲解来源：** [同名逐章意见](../../comment/029_variable_windows.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

这一章同时解决两个经典问题，它们共用同一套"滑动窗口"骨架，但优化方向完全相反。

**问题 A（最长无重复子串）**：给你一个字符串 s，你要找出其中最长的一段连续子串，要求这段子串里没有任何字符重复。具体例子：输入 `s='abcabcbb'`，输出 `3`。为什么是 3？因为整个串里最长的无重复片段是 `'abc'`（出现在下标 0~2），长度 3；你往后不管怎么取，比如 `'abca'` 就有两个 `a`，不合法。

**问题 B（最短覆盖子串）**：给你字符串 s 和目标串 t，你要在 s 里找一段**最短**的连续子串，使得 t 里每个字符（含重复）都被这段覆盖至少同样次数。具体例子：输入 `s='ADOBECODEBANC'`、`t='ABC'`，输出 `'BANC'`。为什么是 `'BANC'`？因为这四个字母中包含 A、B、C，能够覆盖 `t='ABC'`；而 s 里更短的候选（比如 `'BAN'` 缺 C、`'ANC'` 缺 B）都盖不全，所以 `'BANC'` 就是最短答案。

注意两个问题的"方向"相反：问题 A 想让窗口**尽量长**，一出现重复就收缩；问题 B 想让窗口**尽量短**，一旦盖住了 t 就立刻收缩，看还能缩到多短。

## 2. 基础知识：先读懂这些词

- **可变窗口（滑动窗口）**：用两个下标 `left` 和 `right` 圈住一段连续区间，`right` 不断右移扩大窗口，`left` 视情况右移缩小窗口。你可以把它想象成一把在字符串上滑动的尺子，两头都能动。
- **窗口有效条件**：判断当前窗口是否"合法"的判据。问题 A 的判据是"窗口内每个字符出现次数都不超过 1"；问题 B 的判据是"窗口里 t 的每个字符都够了数量"。写滑动窗口题的第一步永远是把这个条件写清楚。
- **频次表 / Counter**：一个字典，key 是字符，value 是该字符在当前窗口里出现了几次。窗口右移时加一，左移时减一，这样任何时刻都能 O(1) 查询某个字符的出现次数。
- **缺失计数（missing）**：问题 B 专用的一个整数，表示"还差几个字符才能盖住 t"。它等于 t 中尚未被窗口满足的字符个数总和。missing 变成 0 就代表窗口盖住了 t。用它的好处是不用每步都遍历整个 need 字典去比对。
- **计入重复字符**：t 如果是 `'AAB'`，那窗口里必须有两个 A 和一个 B，只统计"出现过哪些字符种类"是不够的，必须按个数记账。这是本章代码用 `Counter(t)` 而不是 `set(t)` 的原因。
- **可行性单调前提**：滑动窗口能工作的隐藏前提——窗口越长越"容易违法"（问题 A）或"容易满足"（问题 B），即合法性随窗口长度单调变化，因此应按各自条件决定收缩时机：无重复窗口在非法时收缩，覆盖窗口在合法时收缩。如果约束不单调（比如带负数的和），这套收缩逻辑就会失效。

## 3. 思路推导：从小例子开始

先区分两个问题的目标，不能只记一份窗口模板。

**最长无重复子串。** `counts[c]` 是窗口内字符 `c` 的个数。右端读入字符后，如果该字符重复，就依次移走左端字符；恢复无重复后，才用 `right-left+1` 更新最长长度。例如 `abcaac` 的合法窗口依次是 `a`、`ab`、`abc`、`bca`、`a`、`ac`，最长为 3。读入第二个连续的 `a` 时，仅移走 `b` 不能消除重复，需要继续移走 `c` 和旧的 `a`。

**最短覆盖子串。** `need[c]` 不是窗口频次，而是“目标需要的次数减去窗口中已有的次数”；负值表示有富余。`missing` 是尚未满足的字符总个数，包含目标中的重复字符。右端加入一个仍然缺少的字符时，`missing` 减一。它变成 0 后，先保存候选答案，再试着移走左端字符；一旦移走必需字符，停止收缩。

以 `s='ADOBECODEBANC'`、`t='ABC'` 为例，统一用左闭右开区间 `[left, end)`：

| 右端位置 | 仍能覆盖的最短候选 | 长度 | 再移走哪个字符会失去覆盖 |
|---|---|---:|---|
| 5，读到 `C` | `[0,6)` = `ADOBEC` | 6 | 下标 0 的 `A` |
| 10，读到 `A` | `[5,11)` = `CODEBA` | 6 | 下标 5 的 `C` |
| 12，读到 `C` | `[9,13)` = `BANC` | 4 | 下标 9 的 `B` |

因此答案是 `BANC`，长度 **4**。`[10,13)` 是 `ANC`，长度 3，却缺少 `B`，不能记为答案。下方的运行轨迹会用实际计数状态核对这三次候选。

两个问题都只让左右端点向前移动，但更新答案的条件不同。保存答案下标，最后只切片一次，避免在收缩循环中反复复制字符串。

## 4. 核心代码：length_of_longest_substring

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def length_of_longest_substring(s):
    counts = Counter()
    left = best = 0
    for right, c in enumerate(s):
        counts[c] += 1
        while counts[c] > 1:
            counts[s[left]] -= 1
            left += 1
        best = max(best, right - left + 1)
    return best
```

### 逐段读懂代码

先看 `length_of_longest_substring`，对应实现如下：

```python
from collections import Counter

def length_of_longest_substring(s):
    counts = Counter()
    left = best = 0
    for right, c in enumerate(s):
        counts[c] += 1
        while counts[c] > 1:
            counts[s[left]] -= 1
            left += 1
        best = max(best, right - left + 1)
    return best
```

- `counts=Counter(); left=best=0`：counts 是空频次表；left 是窗口左端；best 记录历史最长长度，初始 0（空串答案就是 0）。
- `for right,c in enumerate(s)`：right 从 0 走到末尾，c 是当前读入的字符。
- `counts[c]+=1`：新字符进窗口，先记账。
- `while counts[c]>1:`：刚进来的 c 出现了两次，窗口非法。注意这里只检查 c 一个字符就够了，因为除了 c 之外的字符在上一轮结束时都没重复。
- `counts[s[left]]-=1; left+=1`：把左端字符移出窗口并右移 left，一行干两件事，循环直到 c 不再重复。
- `best=max(best,right-left+1)`：此时窗口是"以 right 结尾的最长合法子串"，拿它的长度更新答案。

再看 `min_window`，对应实现如下：

```python
def min_window(s, t):
    if not t:
        return ''
    need = Counter(t)
    missing = len(t)
    left = 0
    best = None
    for right, c in enumerate(s):
        if need[c] > 0:
            missing -= 1
        need[c] -= 1
        while missing == 0:
            if best is None or right - left + 1 < best[1] - best[0]:
                best = (left, right + 1)
            old = s[left]
            need[old] += 1
            left += 1
            if need[old] > 0:
                missing += 1
    return '' if best is None else s[best[0]:best[1]]
```

- `if not t: return ''`：目标串为空串时直接返回空串，挡住无意义的输入。
- `need=Counter(t); missing=len(t)`：need 记录每个字符还"欠"几个；missing 是欠账总数。注意 `Counter` 查不存在的 key 会返回 0 而不报错，所以后面 `need[c]` 对 s 特有的字符也能安全访问。
- `if need[c]>0: missing-=1`：如果 c 正好是还欠着的字符，这笔进账能抵一元欠款。
  - 注意判断条件是 `need[c]>0` 而不是 `need[c]`：need[c] 可能是 0（刚好够）或负数（多余），这两种情况下进一个 c 都不能减少 missing。
- `need[c]-=1`：不管是不是欠着的字符都记一笔。对 s 特有的字符，need[c] 会从 0 变 -1，表示"多余一个"。
  - 这一行和上一行顺序不能换：必须先判断"进来的这个 c 能不能抵欠款"，再统一减一。
- `while missing==0:`：欠款还清，窗口合法。进入"边收缩边记录"的循环。
- `if best is None or right-left+1<best[1]-best[0]: best=(left,right+1)`：当前窗口比历史最短更短时，只记录下标对 `(left, right+1)`，注意右端存的是开区间端点 right+1，方便切片。第一次找到合法窗口时 best 还是 None，所以有 `best is None` 这个分支。
- `old=s[left]; need[old]+=1; left+=1`：把左端字符移出、记账、左移。如果移出后 `need[old]>0`，说明我们把一个必需字符丢掉了，欠款重新出现。
- `if need[old]>0: missing+=1`：欠款加一，下一轮 `while missing==0` 条件不成立，自然退出收缩。
- `return '' if best is None else s[best[0]:best[1]]`：从头到尾都没盖住过 t 就返回空串，否则按记录的下标切片返回。

## 5. 分段实现：按顺序运行

**本章接口：** `length_of_longest_substring(s)`、`min_window(s, t)`

### 导入本章使用的库

In [1]:
from collections import Counter

### length_of_longest_substring

In [2]:
def length_of_longest_substring(s):
    counts = Counter()
    left = best = 0
    for right, c in enumerate(s):
        counts[c] += 1
        while counts[c] > 1:
            counts[s[left]] -= 1
            left += 1
        best = max(best, right - left + 1)
    return best

### min_window

In [3]:
def min_window(s, t):
    if not t:
        return ''
    need = Counter(t)
    missing = len(t)
    left = 0
    best = None
    for right, c in enumerate(s):
        if need[c] > 0:
            missing -= 1
        need[c] -= 1
        while missing == 0:
            if best is None or right - left + 1 < best[1] - best[0]:
                best = (left, right + 1)
            old = s[left]
            need[old] += 1
            left += 1
            if need[old] > 0:
                missing += 1
    return '' if best is None else s[best[0]:best[1]]

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [4]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

s='abcaac'; counts=Counter(); left=0; rows=[]
for right,c in enumerate(s):
    counts[c]+=1
    while counts[c]>1:
        counts[s[left]]-=1; left+=1
    rows.append((left,right,s[left:right+1],right-left+1))
show_table(['left','right','合法窗口','长度'],rows)
def trace_min_window(s, t):
    """记录真实合法候选；返回答案及 (右端, 左端, 右开端, 子串, 缺失数)。"""
    if not t:
        return '', []
    need = Counter(t)
    missing = len(t)
    left = 0
    best = None
    rows = []
    for right, char in enumerate(s):
        if need[char] > 0:
            missing -= 1
        need[char] -= 1
        while missing == 0:
            rows.append((right, left, right + 1, s[left:right + 1], missing))
            if best is None or right + 1 - left < best[1] - best[0]:
                best = (left, right + 1)
            removed = s[left]
            need[removed] += 1
            left += 1
            if need[removed] > 0:
                missing += 1
    return ('' if best is None else s[best[0]:best[1]]), rows

answer, trace_rows = trace_min_window('ADOBECODEBANC', 'ABC')
assert answer == min_window('ADOBECODEBANC', 'ABC') == 'BANC'
assert len(answer) == 4
assert (12, 9, 13, 'BANC', 0) in trace_rows
assert all(Counter('ABC') <= Counter(row[3]) for row in trace_rows)
show_table(['右端下标', 'left', 'end（不含）', '合法候选', 'missing'], trace_rows)

left,right,合法窗口,长度
0,0,a,1
0,1,ab,2
0,2,abc,3
1,3,bca,3
4,4,a,1
4,5,ac,2


右端下标,left,end（不含）,合法候选,missing
5,0,6,ADOBEC,0
10,1,11,DOBECODEBA,0
10,2,11,OBECODEBA,0
10,3,11,BECODEBA,0
10,4,11,ECODEBA,0
10,5,11,CODEBA,0
12,6,13,ODEBANC,0
12,7,13,DEBANC,0
12,8,13,EBANC,0
12,9,13,BANC,0


## 7. 正确性、适用条件与复杂度

每个右端点只向前走，左端点也不回退。无重复窗口收缩后是该右端点的最长合法后缀；覆盖窗口循环检查了该右端点所有可覆盖的左端位置，保留最短者。

**代价：** 平均 O(|s|+|t|) 时间、O(σ) 计数空间；保存最佳索引避免循环内复制答案。

### 展开理解

**最长无重复子串。** 处理当前右端字符后，若出现重复就左移，直到再次无重复。右端相同的更短窗口不可能给出更大的长度；更长窗口则已经含重复。因此这一轮记录的是以当前右端结尾的最长合法窗口。枚举全部右端就不会漏掉全局答案。

**最短覆盖子串。** 每当 `missing == 0` 时，先记录候选，再移走左端字符。不能说每轮都重新枚举了所有历史左端点：已经丢弃的左端不会再回来。为什么仍然不漏最优解？因为丢弃某个左端时，已经记录过它对应的合法窗口；以后固定这个左端再扩大右端，只会更长，不可能改善最短答案。若 `missing > 0`，收缩只会减少字符，不会把缺失变成满足，所以应继续扩张。

**代价。** 右端最多前进 |s| 次，左端也最多前进 |s| 次。频次表操作按平均 O(1) 计时，算法总时间是 O(|s|+|t|)，不是两层循环就必然 O(|s|²)。空间为 O(σ)，σ 是实际存入频次表的不同字符数。实现只保存最佳边界，最后切片生成答案。

**可视化的额外代价。** 本章演示版会保存多个子串快照，便于观察状态。这会额外复制字符串，最坏可能需要 O(|s|²) 的记录时间和空间；上面的线性复杂度只属于返回最终答案的 `min_window`，不属于轨迹收集器。

## 8. 单元测试：每个用例在检查什么

“自动测试”部分 的断言逐条解释：

- `assert length_of_longest_substring('abcabcbb')==3`：正常值测试。LeetCode 3 的样例，答案 `'abc'` 长 3，验证主逻辑没跑偏。
- `assert length_of_longest_substring('')==0`：边界值测试。空串一个字符都没有，best 从 0 出发从未被更新，答案应为 0，挡住"空输入返回垃圾值"的实现。
- `assert min_window('ADOBECODEBANC','ABC')=='BANC'`：正常值测试。LeetCode 76 的样例，覆盖了"找到第一个合法窗口后还要继续扫描、后面出现更短答案"的完整流程。
- `assert min_window('a','aa')==''`：特殊值测试。t 需要**两个** a 而 s 只有一个，考察你是否正确处理目标里的重复字符——如果实现只看字符种类（用 set），这里就会错误地返回 `'a'`。
- 后面的大循环：用 `itertools.product` 枚举长度 0~5 的全部 `ab` 二进制串（共 2^0+...+2^5=63 个），对每个串和四种目标 t（`'a','ab','aa','bba'`），用暴力枚举所有子串算出参考答案，再和本章实现对拍。拆开看：
  - `ref=max((r-l for l in range(n+1) for r in range(l,n+1) if len(set(s[l:r]))==r-l),default=0)`：暴力求最长无重复——枚举所有子串，`len(set(...))==r-l` 表示"子串长度等于不同字符数"，即无重复；`default=0` 处理连空串都枚举不到合法解的情况（n=0）。
  - `candidates=[s[l:r] for ... if not (Counter(t)-Counter(s[l:r]))]`：暴力找覆盖子串。`Counter(t)-Counter(win)` 是"t 比窗口多出来的部分"，结果为空（假值取反为真）表示窗口盖住了 t。
  - `expected=min(candidates,key=len) if candidates else ''`：最短覆盖的参考答案；一个都没有就是空串。
  - `assert len(ans)==len(expected)` 只比长度不比内容：因为同样最短的可能有多个，比内容会误杀正确实现；后面 `if ans: assert not (Counter(t)-Counter(ans))` 再单独确认返回的串确实覆盖 t。
  - 四种 t 里特意放了 `'aa'` 和 `'bba'`，都是带重复字符的目标，保证"按个数记账"这个点被反复测到。

In [5]:
assert length_of_longest_substring('abcabcbb') == 3

assert length_of_longest_substring('') == 0

assert min_window('ADOBECODEBANC', 'ABC') == 'BANC'

assert min_window('a', 'aa') == ''

from itertools import product

for n in range(6):
    for s in map(''.join, product('ab', repeat=n)):
        ref = max((r - l for l in range(n + 1) for r in range(l, n + 1) if len(set(s[l:r])) == r - l), default=0)
        assert length_of_longest_substring(s) == ref
        for t in ['a', 'ab', 'aa', 'bba']:
            candidates = [s[l:r] for l in range(n) for r in range(l + 1, n + 1) if not Counter(t) - Counter(s[l:r])]
            expected = min(candidates, key=len) if candidates else ''
            ans = min_window(s, t)
            assert len(ans) == len(expected)
            if ans:
                assert not Counter(t) - Counter(ans)

assert min_window('ADOBECODEBANC', 'ABC') == 'BANC'
assert len(min_window('ADOBECODEBANC', 'ABC')) == 4
assert min_window('aab', 'ab') == 'ab'
assert min_window('abc', '') == ''

print('N029: 所有本章断言通过')

N029: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 构造只用集合无法处理重复目标字符的反例。

**练习 2：** 解释非负和窗口的适用条件。

<details>
<summary>展开思路提示（不是完整答案）</summary>

- **练习 1（构造只用集合无法处理重复目标字符的反例）**：提示：让 t 含有同一个字符两次，而 s 里这个字符的两个出现位置隔得很开。比如 `t='aa'`、`s='ab_a...'`。你先手算正确答案（必须包含两个 a 的最短子串），再说明"只统计字符种类的集合版"为什么会返回一个只含一个 a 的错误窗口。验证时可以写三行暴力：枚举 s 的所有子串，用 `Counter(t)-Counter(win)` 判断覆盖，取最短。
- **练习 2（解释非负和窗口的适用条件）**：提示：想象约束从"无重复"换成"窗口内数字之和不超过 target"。如果数组全是非负数，窗口变长和只会变大或不变，所以违反后不断移出左端一定能让和降回限额以内——这就是收缩能停下来的保证。然后想一个带负数的输入，比如 `[5, -10, 5]`、target 取某个值，手算演示"移出左端后和反而变大/缩小左端跳过了更优解"的现象，说明单调性断了这套逻辑就失效。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [6]:
# 完整实现：本单元格不依赖其他单元格。
from collections import Counter

def length_of_longest_substring(s):
    counts = Counter()
    left = best = 0
    for right, c in enumerate(s):
        counts[c] += 1
        while counts[c] > 1:
            counts[s[left]] -= 1
            left += 1
        best = max(best, right - left + 1)
    return best

def min_window(s, t):
    if not t:
        return ''
    need = Counter(t)
    missing = len(t)
    left = 0
    best = None
    for right, c in enumerate(s):
        if need[c] > 0:
            missing -= 1
        need[c] -= 1
        while missing == 0:
            if best is None or right - left + 1 < best[1] - best[0]:
                best = (left, right + 1)
            old = s[left]
            need[old] += 1
            left += 1
            if need[old] > 0:
                missing += 1
    return '' if best is None else s[best[0]:best[1]]

# 示例与回归测试
assert length_of_longest_substring('abcabcbb') == 3

assert length_of_longest_substring('') == 0

assert min_window('ADOBECODEBANC', 'ABC') == 'BANC'

assert min_window('a', 'aa') == ''

from itertools import product

for n in range(6):
    for s in map(''.join, product('ab', repeat=n)):
        ref = max((r - l for l in range(n + 1) for r in range(l, n + 1) if len(set(s[l:r])) == r - l), default=0)
        assert length_of_longest_substring(s) == ref
        for t in ['a', 'ab', 'aa', 'bba']:
            candidates = [s[l:r] for l in range(n) for r in range(l + 1, n + 1) if not Counter(t) - Counter(s[l:r])]
            expected = min(candidates, key=len) if candidates else ''
            ans = min_window(s, t)
            assert len(ans) == len(expected)
            if ans:
                assert not Counter(t) - Counter(ans)

assert min_window('ADOBECODEBANC', 'ABC') == 'BANC'
assert len(min_window('ADOBECODEBANC', 'ABC')) == 4
assert min_window('aab', 'ab') == 'ab'
assert min_window('abc', '') == ''

print('N029: 所有本章断言通过')

N029: 所有本章断言通过


## 11. 代表题与迁移

- **3. Longest Substring Without Repeating Characters**：这题就是本章的 `length_of_longest_substring` 本体，练的是"频次表 + 重复即收缩"的最长窗口套路。
- **76. Minimum Window Substring**：这题就是本章的 `min_window` 本体，练的是"missing 计数 + 满足即收缩取最短"的反向窗口，以及目标含重复字符时必须按个数记账。

**题面入口：** [3](https://leetcode.com/problems/longest-substring-without-repeating-characters/)

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。